<a href="https://colab.research.google.com/github/malsony791-maker/uni-chat-bot-/blob/main/University_AI_Chatbot_Working_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# University AI Chatbot


In [ ]:
%pip -q uninstall -y torchao
%pip -q install "transformers>=4.45,<5" "peft>=0.13" accelerate safetensors "gradio>=5,<7" pandas openpyxl
from google.colab import drive
drive.mount('/content/drive')
import os
BASE='/content/drive/MyDrive/University_AI_Project'
EXCEL_PATH=f'{BASE}/University_Chatbot_16_Groups.xlsx'
MODEL_PATH=f'{BASE}/Qwen_Training/Qwen_University_Final'
assert os.path.isfile(EXCEL_PATH), 'Excel file is missing from Google Drive'
assert os.path.isfile(f'{MODEL_PATH}/adapter_config.json'), 'LoRA adapter missing'
print('✅ Google Drive, Excel and adapter found')

Mounted at /content/drive
✅ Google Drive, Excel and adapter found


In [ ]:
import os, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
with open(f'{MODEL_PATH}/adapter_config.json', encoding='utf-8') as f:
    BASE_MODEL=json.load(f)['base_model_name_or_path']
print('Base model:',BASE_MODEL, '| GPU:',torch.cuda.is_available())
tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH)
base_model=AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)
model=PeftModel.from_pretrained(base_model,MODEL_PATH)
model.eval()
print('✅ Trained Qwen LoRA loaded (schedule answers remain Excel-grounded)')

Base model: Qwen/Qwen2.5-1.5B-Instruct | GPU: False


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Trained Qwen LoRA loaded (schedule answers remain Excel-grounded)


In [ ]:

import os
import re
import json
import torch
import pandas as pd
import unicodedata

# ==================================================
# UNIVERSITY AI CHATBOT - COMPLETE ENGINE
# Fine-tuned Qwen + Excel + Conflict Detection
# ==================================================

EXCEL_PATH = (
    "/content/drive/MyDrive/University_AI_Project/"
    "University_Chatbot_16_Groups.xlsx"
)

assert os.path.isfile(EXCEL_PATH), "Excel file not found!"
assert "model" in globals() and model is not None, "Run Cell 2 first!"
assert "tokenizer" in globals() and tokenizer is not None, "Tokenizer missing!"
assert hasattr(model, "peft_config"), "Fine-tuned LoRA adapter not loaded!"

model.eval()

AI_STATUS = "Fine-tuned Qwen Parser"

DAYS = {
    "Saturday": 0,
    "Sunday": 1,
    "Monday": 2,
    "Tuesday": 3,
    "Wednesday": 4,
    "Thursday": 5,
    "Friday": 6
}

DAY_ALIASES = {
    "السبت": "Saturday",
    "الاحد": "Sunday",
    "الاثنين": "Monday",
    "الاتنين": "Monday",
    "الثلاثاء": "Tuesday",
    "التلات": "Tuesday",
    "الاربعاء": "Wednesday",
    "الخميس": "Thursday",
    "الجمعه": "Friday"
}

COURSE_ALIASES = {
    "CF216": [
        "data structures", "data structure",
        "داتا ستراكشر", "هياكل البيانات"
    ],
    "CF225": [
        "computer architecture", "architecture",
        "معماريه الحاسب"
    ],
    "BS214": [
        "probability", "statistics",
        "احتمالات", "احصاء"
    ],
    "BS212": [
        "mathematics", "math", "maths",
        "fourier", "differential",
        "رياضيات", "ماث"
    ],
    "BS213": [
        "project management",
        "اداره المشاريع", "اداره المشروعات"
    ],
    "UR211": [
        "communication skills",
        "communication", "مهارات التواصل"
    ]
}

PERSON_ALIASES = {
    "shimaa": ["شيماء", "شيما", "shymaa"],
    "noor": ["نور", "nour"],
    "ahmed": ["احمد"],
    "romany": ["روماني"],
    "doaa": ["دعاء"],
    "mennallah": ["منه الله", "منه"],
    "aya": ["ايه", "ايه حسن"],
    "mohammed": ["محمد", "mohamed", "muhammad"],
    "muhammad": ["محمد", "mohamed", "mohammed"],
    "khaled": ["خالد"],
    "sherin": ["شيرين"],
    "fady": ["فادي"],
    "hedia": ["هديه"]
}

SYSTEM_PROMPT = """You are a university schedule question parser.
Understand Arabic Egyptian and English questions.

Extract these fields:
group, subject, class_type, preferred_day, intent.

Valid intents:
find_alternative
show_schedule
find_instructor
check_conflict

Class types:
LEC = Lecture
TU = Tutorial
LAB = Laboratory

Return JSON only.
Use null for missing information.
Do not invent schedules, instructors or rooms."""

VALID_INTENTS = {
    "find_alternative",
    "show_schedule",
    "find_instructor",
    "check_conflict"
}

VALID_TYPES = {"LEC", "TU", "LAB"}


# ==================================================
# 1. TEXT NORMALIZATION
# ==================================================

def norm(value):
    text = unicodedata.normalize(
        "NFKC", str(value if value is not None else "")
    ).casefold()

    text = re.sub(r"[\u064b-\u065f\u0640]", "", text)
    text = re.sub(r"[أإآ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    text = re.sub(r"[^\w\s]", " ", text)

    return " ".join(text.split())


def found(query, phrase):
    p = norm(phrase)

    return bool(
        p and re.search(
            r"(?<!\w)" + re.escape(p) + r"(?!\w)",
            norm(query)
        )
    )


def has(query, phrases):
    return any(found(query, p) for p in phrases)


def lecture_group(group):
    return "ABCD"[(int(group) - 1) // 4]


def type_matches(value, wanted):
    types = {
        x.strip().upper()
        for x in str(value).split(",")
    }
    return str(wanted).upper() in types


def normalize_time(value):
    if pd.isna(value):
        return None

    if hasattr(value, "strftime"):
        return value.strftime("%H:%M")

    text = str(value).strip()

    try:
        return pd.to_datetime(text).strftime("%H:%M")
    except Exception:
        match = re.search(r"(\d{1,2}):(\d{2})", text)

        if match:
            return (
                f"{int(match.group(1)):02}:"
                f"{match.group(2)}"
            )

    return None


# ==================================================
# 2. LOAD VERIFIED EXCEL
# ==================================================

def load_excel(path):
    df = pd.read_excel(
        path,
        sheet_name="All Classes",
        engine="openpyxl"
    )

    required = [
        "Group", "Day", "Start Time", "End Time",
        "Course Code", "Course Name", "Class Type",
        "Instructor / TA", "Room / Location", "QA Status"
    ]

    missing = [
        column for column in required
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            "Missing Excel columns: " + ", ".join(missing)
        )

    df = df.copy()

    df["Group"] = pd.to_numeric(
        df["Group"], errors="coerce"
    )

    df = df[df["Group"].between(1, 16)]

    df = df[
        df["QA Status"].astype(str)
        .str.strip().str.casefold()
        .eq("parsed")
    ]

    df["Start Time"] = df["Start Time"].apply(normalize_time)
    df["End Time"] = df["End Time"].apply(normalize_time)

    df = df.dropna(
        subset=[
            "Group", "Course Code", "Course Name",
            "Start Time", "End Time"
        ]
    )

    df["Group"] = df["Group"].astype(int)
    df = df.fillna("")

    return df.to_dict("records")


records = load_excel(EXCEL_PATH)
data = pd.DataFrame(records)

assert len(records) > 0, "No verified schedule records!"

print("✅ Excel records loaded:", len(records))


# ==================================================
# 3. COURSE / DAY / TYPE DETECTION
# ==================================================

def get_subject(query):
    q = norm(query)

    courses = data[
        ["Course Code", "Course Name"]
    ].drop_duplicates()

    for _, row in courses.iterrows():
        code = str(row["Course Code"]).upper()
        name = str(row["Course Name"])

        if found(q, code) or found(q, name):
            return code, name

    for _, row in courses.iterrows():
        code = str(row["Course Code"]).upper()

        if has(q, COURSE_ALIASES.get(code, [])):
            return code, str(row["Course Name"])

    return None, None


def get_day(query):
    q = norm(query)

    for alias, day in DAY_ALIASES.items():
        if found(q, alias):
            return day

    for day in DAYS:
        if found(q, day):
            return day

    return None


def get_type(query):
    q = norm(query)

    if re.search(
        r"محاضر|محاضرات|lecture|\blec\b", q
    ):
        return "LEC"

    if re.search(
        r"سكشن|سكاشن|section|tutorial|\btu\b", q
    ):
        return "TU"

    if re.search(
        r"معمل|معامل|\blab\b|laboratory", q
    ):
        return "LAB"

    return None


# ==================================================
# 4. QWEN PARSER
# ==================================================

def parse_with_qwen(question):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": str(question)
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    try:
        result = json.loads(response)

    except json.JSONDecodeError:
        match = re.search(r"\{[\s\S]*?\}", response)

        if not match:
            raise ValueError("Qwen returned invalid JSON.")

        result = json.loads(match.group(0))

    if not isinstance(result, dict):
        raise ValueError("Qwen output must be an object.")

    return result


def validate_parser(parsed, selected_group):
    try:
        group = int(parsed.get("group"))
    except (TypeError, ValueError):
        group = int(selected_group)

    if group not in range(1, 17):
        group = int(selected_group)

    subject = parsed.get("subject")
    code, title = (None, None)

    if isinstance(subject, str) and subject.strip():
        code, title = get_subject(subject)

    typ = parsed.get("class_type")

    if isinstance(typ, str):
        typ = typ.upper().strip()

    if typ not in VALID_TYPES:
        typ = None

    raw_day = parsed.get("preferred_day")
    day = get_day(raw_day) if raw_day else None

    intent = parsed.get("intent")

    if intent not in VALID_INTENTS:
        intent = None

    return {
        "group": group,
        "course_code": code,
        "subject": title,
        "class_type": typ,
        "preferred_day": day,
        "intent": intent
    }


# ==================================================
# 5. CONFLICT DETECTION
# ==================================================

def overlap(a, b):
    return (
        norm(a["Day"]) == norm(b["Day"])
        and a["Start Time"] < b["End Time"]
        and a["End Time"] > b["Start Time"]
    )


def alternatives(
    group, course_code, class_type,
    records, preferred_day=None
):
    own = [
        r for r in records
        if int(r["Group"]) == int(group)
    ]

    matches = [
        r for r in records
        if str(r["Course Code"]).upper() == course_code
        and type_matches(r["Class Type"], class_type)
    ]

    sessions = {}

    for r in matches:
        g = int(r["Group"])

        if class_type == "LEC":
            key = (
                course_code,
                norm(r["Day"]),
                r["Start Time"],
                r["End Time"],
                norm(r["Room / Location"])
            )
        else:
            key = (
                g, course_code, str(r["Class Type"]),
                norm(r["Day"]), r["Start Time"],
                r["End Time"], norm(r["Room / Location"])
            )

        if key not in sessions:
            sessions[key] = {
                "row": r,
                "groups": set(),
                "lecture_groups": set()
            }

        sessions[key]["groups"].add(g)
        sessions[key]["lecture_groups"].add(
            lecture_group(g)
        )

    own_session_keys = set()

    for item in sessions.values():
        if int(group) in item["groups"]:
            r = item["row"]

            own_session_keys.add((
                norm(r["Day"]),
                r["Start Time"],
                r["End Time"],
                norm(r["Room / Location"])
            ))

    output = []

    for item in sessions.values():
        r = item["row"]

        session_key = (
            norm(r["Day"]),
            r["Start Time"],
            r["End Time"],
            norm(r["Room / Location"])
        )

        # Exclude the student's existing session
        if session_key in own_session_keys:
            continue

        if preferred_day and norm(r["Day"]) != norm(preferred_day):
            continue

        conflicts = []
        seen = set()

        for current in own:
            if not overlap(r, current):
                continue

            # The original session being replaced is not a clash
            if (
                str(current["Course Code"]).upper() == course_code
                and type_matches(current["Class Type"], class_type)
            ):
                continue

            key = (
                current["Course Code"],
                current["Class Type"],
                current["Day"],
                current["Start Time"],
                current["End Time"]
            )

            if key not in seen:
                seen.add(key)
                conflicts.append(current)

        output.append({
            "row": r,
            "groups": sorted(item["groups"]),
            "lecture_groups": sorted(item["lecture_groups"]),
            "conflicts": conflicts
        })

    return sorted(
        output,
        key=lambda item: (
            len(item["conflicts"]) > 0,
            DAYS.get(str(item["row"]["Day"]), 99),
            item["row"]["Start Time"]
        )
    )


# ==================================================
# 6. FORMAT ALTERNATIVES
# ==================================================

def display_results(
    results, subject, typ, group, day=None
):
    if not results:
        return (
            f"مش لاقي بدائل مؤكدة لـ "
            f"{subject} ({typ}) في الجدول."
        )

    free = sum(
        1 for x in results if not x["conflicts"]
    )

    lines = [
        f"### بدائل {subject} ({typ}) — Group {group}",
        (
            f"**الإجمالي: {len(results)}** | "
            f"بدون تعارض: **{free}** | "
            f"بتعارض: **{len(results)-free}**"
        ),
        ""
    ]

    for i, item in enumerate(results, 1):
        row = item["row"]

        if typ == "LEC":
            target = "Lecture " + ", ".join(
                item["lecture_groups"]
            )
        else:
            target = "Group " + ", ".join(
                map(str, item["groups"])
            )

        lines.append(
            f"**{i}. {target}** — "
            f"{row['Day']} | "
            f"{row['Start Time']}–{row['End Time']} | "
            f"{row['Room / Location']}"
        )

        if item["conflicts"]:
            clashes = "; ".join(
                f"{c['Course Name']} "
                f"({c['Class Type']}) "
                f"{c['Start Time']}–{c['End Time']}"
                for c in item["conflicts"]
            )

            lines.append(
                f"⚠️ **تعارض:** {clashes}"
            )
        else:
            lines.append("✅ **مفيش تعارض مكتشف**")

    lines.append(
        "\n> النتائج من السجلات المؤكدة فقط، "
        "والتبديل قد يحتاج موافقة الجامعة."
    )

    return "\n\n".join(lines)


# ==================================================
# 7. SEARCH INSTRUCTORS AND SCHEDULES
# ==================================================

def staff_names(value):
    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]


def get_person(question):
    q = norm(question)
    candidates = []

    for cell in data["Instructor / TA"].unique():
        for person in staff_names(cell):
            p = norm(person)

            words = [
                w for w in p.split()
                if w not in ("dr", "ta", "doctor")
                and len(w) >= 3
            ]

            score = 0

            for word in words:
                aliases = [word] + PERSON_ALIASES.get(
                    word, []
                )

                if has(q, aliases):
                    score += 1

            if score:
                candidates.append(
                    (score, len(words), person)
                )

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: (x[0], x[0] / max(x[1], 1)),
        reverse=True
    )

    top_score = candidates[0][0]

    tied = {
        norm(c[2]) for c in candidates
        if c[0] == top_score
    }

    if len(tied) > 1:
        return None

    return candidates[0][2]


def rows_for(
    group=None, day=None, typ=None,
    code=None, person=None
):
    df = data.copy()

    if group is not None:
        df = df[
            pd.to_numeric(
                df["Group"], errors="coerce"
            ).eq(int(group))
        ]

    if day:
        df = df[
            df["Day"].astype(str)
            .map(norm).eq(norm(day))
        ]

    if typ:
        df = df[
            df["Class Type"].astype(str).apply(
                lambda x: type_matches(x, typ)
            )
        ]

    if code:
        df = df[
            df["Course Code"].astype(str)
            .str.upper().eq(str(code).upper())
        ]

    if person:
        df = df[
            df["Instructor / TA"].astype(str).apply(
                lambda cell: any(
                    norm(x) == norm(person)
                    for x in staff_names(cell)
                )
            )
        ]

    df = df.copy()

    df["_day_order"] = df["Day"].map(
        lambda x: DAYS.get(str(x), 99)
    )

    df = df.sort_values(
        ["_day_order", "Start Time"]
    )

    return df.drop(columns=["_day_order"])


def pretty_rows(df, limit=30):
    if df.empty:
        return "مش لاقي حصص مطابقة في الجدول المؤكد."

    lines = []

    for _, r in df.head(limit).iterrows():
        instructor = (
            str(r.get("Instructor / TA", "")).strip()
            or "غير مذكور"
        )

        lines.append(
            f"- **{r['Course Name']} "
            f"({r['Class Type']})** — "
            f"{r['Day']} "
            f"{r['Start Time']}–{r['End Time']} | "
            f"Group {r['Group']} | "
            f"{r['Room / Location']} | "
            f"{instructor}"
        )

    if len(df) > limit:
        lines.append(
            f"وفيه {len(df)-limit} سجل إضافي."
        )

    return "\n".join(lines)


def teacher_summary(code, typ=None, role=None):
    subset = rows_for(code=code, typ=typ)

    if role == "doctor":
        subset = subset[
            subset["Class Type"].apply(
                lambda x: type_matches(x, "LEC")
            )
        ]

    elif role == "ta":
        subset = subset[
            subset["Class Type"].apply(
                lambda x: (
                    type_matches(x, "TU")
                    or type_matches(x, "LAB")
                )
            )
        ]

    names = {}

    for _, r in subset.iterrows():
        for person in staff_names(r["Instructor / TA"]):
            key = norm(person)

            if key not in names:
                names[key] = {
                    "name": person,
                    "types": set()
                }

            names[key]["types"].add(
                str(r["Class Type"])
            )

    if not names:
        return "مفيش أسماء مسجلة في البيانات المؤكدة."

    return "\n".join(
        f"- **{item['name']}** — "
        f"{', '.join(sorted(item['types']))}"
        for item in names.values()
    )


# ==================================================
# 8. MAIN CHATBOT ENGINE
# ==================================================

def chat_engine(message, history, student_group):
    try:
        raw = parse_with_qwen(message)

        parsed = validate_parser(raw, student_group)

        group = parsed["group"]
        code = parsed["course_code"]
        subject = parsed["subject"]
        typ = parsed["class_type"]
        day = parsed["preferred_day"]
        intent = parsed["intent"]

        # Fallback for missing parser fields
        if not code:
            code, subject = get_subject(message)

        if not typ:
            typ = get_type(message)

        if not day:
            day = get_day(message)

        q = norm(message)

        if not intent:
            if "بديل" in q or has(q, [
                "alternative", "alternatives", "switch"
            ]):
                intent = "find_alternative"

            elif has(q, [
                "مين", "دكتور", "دكتوره",
                "معيد", "معيده", "who",
                "instructor", "teacher"
            ]):
                intent = "find_instructor"

            else:
                intent = "show_schedule"

        print(
            "QWEN PARSER:",
            json.dumps(raw, ensure_ascii=False)
        )

        # ----------------------------------
        # ALTERNATIVES AND CONFLICTS
        # ----------------------------------

        if intent in (
            "find_alternative", "check_conflict"
        ):
            if not code:
                return "حددي اسم المادة المطلوبة."

            if not typ:
                return (
                    "عايزة بديل لمحاضرة ولا سكشن ولا معمل؟"
                )

            results = alternatives(
                group, code, typ, records, day
            )

            return display_results(
                results, subject, typ, group, day
            )

        # ----------------------------------
        # INSTRUCTORS / TEACHING ASSISTANTS
        # ----------------------------------

        if intent == "find_instructor":
            asks_doctor = has(q, [
                "دكتور", "دكتوره",
                "doctor", "professor"
            ])

            asks_ta = has(q, [
                "معيد", "معيده",
                "teaching assistant", "ta"
            ])

            person = get_person(message)

            if person:
                df = rows_for(
                    person=person,
                    day=day,
                    typ=typ,
                    code=code
                )

                return (
                    f"**حصص {person}:**\n\n"
                    + pretty_rows(df)
                )

            if code:
                role = None

                if asks_doctor and not asks_ta:
                    role = "doctor"

                elif asks_ta and not asks_doctor:
                    role = "ta"

                return (
                    f"**المدرسون المسجلون لمادة "
                    f"{subject}:**\n\n"
                    + teacher_summary(
                        code, typ, role
                    )
                )

            return (
                "حددي اسم المادة أو الدكتور أو المعيد."
            )

        # ----------------------------------
        # SCHEDULE QUESTIONS
        # ----------------------------------

        if intent == "show_schedule":
            df = rows_for(
                group=group,
                day=day,
                typ=typ,
                code=code
            )

            heading = f"**جدول Group {group}"

            if day:
                heading += f" — {day}"

            heading += "**\n\n"

            return heading + pretty_rows(df)

        return "ممكن توضحي السؤال أكتر؟"

    except Exception as exc:
        print(
            "CHATBOT ERROR:",
            type(exc).__name__, str(exc)
        )

        return (
            "حصلت مشكلة أثناء معالجة السؤال:\n"
            f"`{type(exc).__name__}: {str(exc)[:200]}`"
        )


# ==================================================
# 9. READY CHECK
# ==================================================

print("\n===================================")
print("✅ FINE-TUNED QWEN CONNECTED")
print("✅ Excel records:", len(records))
print("✅ Conflict detection ready")
print("✅ Chatbot Engine ready")
print("===================================")


✅ Excel records loaded: 173

✅ FINE-TUNED QWEN CONNECTED
✅ Excel records: 173
✅ Conflict detection ready
✅ Chatbot Engine ready


In [ ]:
import gradio as gr
with gr.Blocks(title='University AI Chatbot') as demo:
    gr.Markdown('# 🎓 University AI Chatbot\nاسألي بالعربي أو الإنجليزي عن الجدول والدكاترة والمعيدين والبدائل — الإجابات من Excel المؤكد.')
    group_input=gr.Dropdown(choices=[str(i) for i in range(1,17)],value='15',label='Your Student Group')
    gr.ChatInterface(fn=chat_engine,additional_inputs=[group_input],cache_examples=False)
    gr.Markdown('**ملاحظة:** البدائل بدون تعارض مكتشف لا تعني موافقة الجامعة. رابط المشاركة مؤقت؛ لا تشاركيه مع بيانات حساسة.')
print('✅ Launching Gradio')
demo.launch(share=True,show_error=True)

✅ Launching Gradio
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://005fd0f9e4d759b756.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
